# YuE2 Studio 5.0.0 — Kaggle assets builder

**Run once with Internet ON and Accelerator None.** This prepares the official Linux runtime for the real [YuE2 Studio](https://github.com/timoncool/YuE2-Studio), plus a checksum-verified Cloudflare Tunnel binary, the matching upstream React frontend build, and a small Node runtime to serve it. It does not generate music or use a GPU.

The builder downloads the official **YuE2 Studio 5.0.0 Linux AppImage** from the v5.0.0 GitHub release and verifies it against that release's published SHA-256 manifest. It also downloads and verifies cloudflared and Node.js, then builds the actual React interface from the same upstream release tag.

## Steps
1. Run this notebook on Kaggle with Internet enabled and **Accelerator: None**.
2. Run both builder code cells. The output folder is `/kaggle/working/yue2_studio_assets`.
3. Use **Save Version** and save the notebook output.
4. In the companion **YuE2 Studio 5.0.0 — Kaggle Studio** notebook, attach this notebook's output using **Add Input → Notebook Output Files**, or publish the output as a private Kaggle Dataset using the optional cell below.

This is not the former ComfyUI cover notebook. The companion notebook starts the upstream Studio 5.0.0 application backend and serves the upstream Studio React interface in a browser through Cloudflare.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import requests
import time

STUDIO_VERSION = "5.0.0"
STUDIO_TAG = "v5.0.0"
CLOUDFLARED_VERSION = "2026.10.3"
OUTPUT = Path("/kaggle/working/yue2_studio_assets")
OUTPUT.mkdir(parents=True, exist_ok=True)

GITHUB_RELEASE_API = f"https://api.github.com/repos/timoncool/YuE2-Studio/releases/tags/{STUDIO_TAG}"
CLOUDFLARED_URL = (
    f"https://github.com/cloudflare/cloudflared/releases/download/"
    f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
)
CLOUDFLARED_SHA256 = "053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0"
MANIFEST_NAME = "yue2-studio-assets.json"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def fetch(url, timeout=(30, 300)):
    response = requests.get(
        url,
        headers={"User-Agent": "JJP-YuE2-Studio-Kaggle-Builder/1.0"},
        timeout=timeout,
    )
    response.raise_for_status()
    return response

def download_verified(url, destination, expected_sha256, expected_size=None):
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, 4):
        try:
            h = hashlib.sha256()
            count = 0
            with requests.get(
                url,
                stream=True,
                headers={"User-Agent": "JJP-YuE2-Studio-Kaggle-Builder/1.0"},
                timeout=(45, 600),
            ) as response:
                response.raise_for_status()
                with open(partial, "wb") as out:
                    for block in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if not block:
                            continue
                        out.write(block)
                        h.update(block)
                        count += len(block)
            digest = h.hexdigest()
            if expected_size is not None and count != int(expected_size):
                raise RuntimeError(
                    f"Size mismatch for {destination.name}: got {count:,}, "
                    f"expected {int(expected_size):,} bytes"
                )
            if digest.lower() != expected_sha256.lower():
                raise RuntimeError(
                    f"SHA-256 mismatch for {destination.name}: {digest}"
                )
            partial.replace(destination)
            return {"name": destination.name, "size": count, "sha256": digest}
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise
            print(f"Download attempt {attempt} failed; retrying...")
            time.sleep(3 * attempt)

print("Checking the official YuE2 Studio release...")
release = fetch(GITHUB_RELEASE_API).json()
if release.get("tag_name") != STUDIO_TAG:
    raise RuntimeError(
        f"Expected release {STUDIO_TAG}, got {release.get('tag_name')!r}"
    )

release_assets = {item["name"]: item for item in release.get("assets", [])}
appimage_name = f"YuE2-Studio-{STUDIO_VERSION}-linux-x86_64.AppImage"
if appimage_name not in release_assets or "SHA256SUMS.txt" not in release_assets:
    raise RuntimeError("The official v5.0.0 AppImage or SHA256SUMS.txt is missing from the release.")

checksums_text = fetch(release_assets["SHA256SUMS.txt"]["browser_download_url"]).text
expected_appimage_sha = None
for line in checksums_text.splitlines():
    parts = line.strip().split()
    if len(parts) >= 2 and Path(parts[-1].lstrip("*")).name == appimage_name:
        expected_appimage_sha = parts[0].lower()
        break
if not expected_appimage_sha or not re.fullmatch(r"[0-9a-f]{64}", expected_appimage_sha):
    raise RuntimeError(f"No valid SHA-256 entry for {appimage_name} in the official release manifest.")

appimage_asset = release_assets[appimage_name]
print(f"Downloading official YuE2 Studio {STUDIO_VERSION} Linux AppImage ({appimage_asset['size']:,} bytes)...")
appimage_info = download_verified(
    appimage_asset["browser_download_url"],
    OUTPUT / appimage_name,
    expected_appimage_sha,
    appimage_asset["size"],
)
(OUTPUT / appimage_name).chmod(0o755)

print(f"Downloading cloudflared {CLOUDFLARED_VERSION}...")
cloudflared_info = download_verified(
    CLOUDFLARED_URL,
    OUTPUT / "cloudflared-linux-amd64",
    CLOUDFLARED_SHA256,
)
(OUTPUT / "cloudflared-linux-amd64").chmod(0o755)

manifest = {
    "schema": 1,
    "studio_name": "YuE2 Studio",
    "studio_version": STUDIO_VERSION,
    "studio_tag": STUDIO_TAG,
    "studio_source": "https://github.com/timoncool/YuE2-Studio",
    "studio_license": "MIT",
    "appimage_url": appimage_asset["browser_download_url"],
    "appimage": appimage_info,
    "release_checksum_manifest": release_assets["SHA256SUMS.txt"]["browser_download_url"],
    "cloudflared_version": CLOUDFLARED_VERSION,
    "cloudflared_url": CLOUDFLARED_URL,
    "cloudflared": cloudflared_info,
}
(OUTPUT / MANIFEST_NAME).write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(OUTPUT / "README.txt").write_text(
    "YuE2 Studio 5.0.0 Kaggle runtime assets\n"
    "Official upstream Linux AppImage and checksum-pinned cloudflared binary.\n"
    "Attach this folder as an input to YuE2_Studio_5_0_0_Kaggle_Studio.ipynb.\n"
    "The Studio downloads its model components on first use from its own Models/Setup interface.\n",
    encoding="utf-8",
)
print("\nVerified assets are ready:")
for item in sorted(OUTPUT.iterdir()):
    print(f"  {item.name:52} {item.stat().st_size:>14,} bytes")
print(f"\nSave this notebook version with outputs. Asset folder: {OUTPUT}")

## Build the matching browser interface

This cell checksums Node.js, checks out the exact upstream v5.0.0 source, installs its locked frontend dependencies, and builds the real Studio React interface. It stores the compiled website and a small Node runtime beside the official Linux AppImage. The runtime notebook therefore does not need to clone the source or run npm ci again.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import requests

NODE_VERSION = "22.14.0"
NODE_ARCHIVE = f"node-v{NODE_VERSION}-linux-x64.tar.xz"
NODE_BASE = f"https://nodejs.org/dist/v{NODE_VERSION}"
SOURCE = Path("/kaggle/temp/yue2-studio-v5-source")
NODE_TEMP = Path("/kaggle/temp/yue2-node")
NODE_TEMP.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("Preparing the official Node.js build toolchain...")
checksums = requests.get(f"{NODE_BASE}/SHASUMS256.txt", timeout=60)
checksums.raise_for_status()
node_sha = None
for line in checksums.text.splitlines():
    fields = line.split()
    if len(fields) >= 2 and fields[-1].lstrip("*") == NODE_ARCHIVE:
        node_sha = fields[0].lower()
        break
if not node_sha or len(node_sha) != 64:
    raise RuntimeError(f"Official Node.js SHA-256 not found for {NODE_ARCHIVE}")

node_archive = NODE_TEMP / NODE_ARCHIVE
if not node_archive.is_file() or file_sha256(node_archive) != node_sha:
    with requests.get(f"{NODE_BASE}/{NODE_ARCHIVE}", stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with open(node_archive, "wb") as output:
            for block in response.iter_content(8 * 1024 * 1024):
                if block:
                    output.write(block)
if file_sha256(node_archive) != node_sha:
    raise RuntimeError("Node.js archive failed its official SHA-256 check")

node_extract = NODE_TEMP / f"node-v{NODE_VERSION}-linux-x64"
if not (node_extract / "bin/node").is_file():
    node_extract.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xJf", str(node_archive), "-C", str(NODE_TEMP)], check=True)
node_binary = node_extract / "bin/node"
if not node_binary.is_file():
    raise RuntimeError("Node.js archive did not contain bin/node")
node_binary.chmod(0o755)
node_version_text = subprocess.check_output([str(node_binary), "--version"], text=True).strip()
if node_version_text != f"v{NODE_VERSION}":
    raise RuntimeError(f"Unexpected Node.js version: {node_version_text}")

print("Cloning the exact YuE2 Studio release source tag...")
if SOURCE.exists():
    shutil.rmtree(SOURCE)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "v5.0.0", "--single-branch",
    "https://github.com/timoncool/YuE2-Studio.git", str(SOURCE)
], check=True)
source_commit = subprocess.check_output(
    ["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True
).strip()
config = json.loads((SOURCE / "desktop/src-tauri/tauri.conf.json").read_text())
if config.get("version") != "5.0.0":
    raise RuntimeError(f"Source tag/version mismatch: {config.get('version')}")

print("Installing pinned frontend dependencies and building the upstream Studio UI...")
env = os.environ.copy()
env["PATH"] = str(node_extract / "bin") + os.pathsep + env.get("PATH", "")
subprocess.run(
    ["npm", "ci", "--no-audit", "--no-fund"],
    cwd=SOURCE / "app", env=env, check=True
)
subprocess.run(
    ["npm", "run", "build"],
    cwd=SOURCE / "app", env=env, check=True
)

dist = SOURCE / "app/dist"
for required in ("index.html", "visualizer.html"):
    if not (dist / required).is_file():
        raise RuntimeError(f"Studio frontend build is missing {required}")

frontend_output = OUTPUT / "studio-web"
if frontend_output.exists():
    shutil.rmtree(frontend_output)
shutil.copytree(dist, frontend_output)

runtime_node = OUTPUT / "node-runtime"
runtime_node.mkdir(parents=True, exist_ok=True)
shutil.copy2(node_binary, runtime_node / "node")
(runtime_node / "node").chmod(0o755)

frontend_files = {}
for path in sorted(frontend_output.rglob("*")):
    if path.is_file():
        frontend_files[str(path.relative_to(frontend_output))] = {
            "size": path.stat().st_size,
            "sha256": file_sha256(path),
        }

manifest_path = OUTPUT / MANIFEST_NAME
manifest = json.loads(manifest_path.read_text())
manifest["frontend"] = {
    "source": "https://github.com/timoncool/YuE2-Studio/tree/v5.0.0",
    "source_commit": source_commit,
    "build_command": "npm ci && npm run build",
    "node_version": NODE_VERSION,
    "files": frontend_files,
}
manifest["node_runtime"] = {
    "version": node_version_text,
    "size": (runtime_node / "node").stat().st_size,
    "sha256": file_sha256(runtime_node / "node"),
}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"\nUpstream UI built from commit {source_commit}.")
print(f"Frontend files: {len(frontend_files)}")
print(f"Frontend size: {sum(x['size'] for x in frontend_files.values()):,} bytes")
print(f"Node runtime: {node_version_text}")
print(f"Assets ready in: {OUTPUT}")

## Optional: publish the assets as a private Kaggle Dataset

Leave `DATASET_HANDLE` as `None` when you attach the saved notebook output directly. To reuse the same assets across notebooks, set a new private dataset slug such as `your-kaggle-username/yue2-studio-5-assets`, configure Kaggle authentication through Kaggle Secrets, and run the next cell. Do not paste API tokens into notebook code.

In [ ]:
DATASET_HANDLE = None  # Example: "your-kaggle-username/yue2-studio-5-assets"

if DATASET_HANDLE:
    import kagglehub
    kagglehub.dataset_upload(
        DATASET_HANDLE,
        str(OUTPUT),
        version_notes="Official YuE2 Studio 5.0.0 Linux runtime and checksum-verified Cloudflare tunnel binary",
    )
    print(f"Private dataset uploaded: https://www.kaggle.com/datasets/{DATASET_HANDLE}")
else:
    print("Skipped. Save the builder notebook output and attach it to the Studio runtime notebook.")